# 01b. Logistic Regression in Practice: Boundaries, Thresholds and Evaluation

**Goals for this notebook**

- Fit logistic regression with **scikit-learn** on 2D data, and read its weights.
- See the **decision boundary** (a straight line) and the **predicted probability** as a colour map.
- Turn probabilities into classes with a **threshold**, and understand what changing it does.
- **Evaluate** a classifier properly: confusion matrix, accuracy, precision, recall, F1, the ROC curve
  and AUC.
- Handle the **general case** of many features: reading weights, and looking at 2D slices.

As before: run every cell in order, do the exercises when you reach them, and use the check cells
(✅ / ❌).

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

from sklearn.linear_model import LogisticRegression

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJXMSI6IDAuODE3NTc0NDc2MTkzNjQzNywgIlcyIjogMC4zNjY5ODQ1ODc1NDAxMDAyLCAiVzMiOiBbMS40LCAtMC43LCAwLjddLCAiMS5CIjogNS4xNjYzMTg1MDMxNzQ0MTcsICIyLkEiOiA2LjYxMDA4ODg5NjE0NDc3MywgIjIuQiI6IFs1LjE2NjMxODUwMzE3NDQxNywgMC40NTQ5NjY1ODk2NDIxNzhdLCAiMy5BIjogeyJhcmdzIjogW1tbMC4yLCAwLjUsIDAuNDksIDAuOSwgMC43XSwgMC41XSwgW1swLjIsIDAuNSwgMC40OSwgMC45LCAwLjddLCAwLjhdXSwgIm91dCI6IFtbMCwgMSwgMCwgMSwgMV0sIFswLCAwLCAwLCAxLCAwXV19LCAiMy5BMiI6IFtbMS4wLCAxLjAsIDEuMCwgMS4wLCAzLjAsIDQuMCwgNS4wLCA2LjAsIDkuMCwgOS4wLCAxMS4wLCAxMy4wLCAxNC4wLCAxOC4wLCAyMC4wLCAyMS4wLCAyNi4wLCAzMC4wLCAzNS4wXSwgWzQzLjAsIDM1LjAsIDI0LjAsIDE5LjAsIDE3LjAsIDE2LjAsIDEyLjAsIDguMCwgNi4wLCA2LjAsIDYuMCwgNi4wLCA1LjAsIDQuMCwgMy4wLCAzLjAsIDIuMCwgMS4wLCAwLjBdXSwgIjMuQiI6IDAuMiwgIjMuQyI6IDQuMjU1NDAwODA0OTIzODY0LCAiNC5BIjogeyJhcmdzIjogW1tbMSwgMCwgMSwgMSwgMCwgMCwgMSwgMCwgMSwgMF0sIFsxLCAwLCAwLCAxLCAxLCAwLCAxLCAwLCAwLCAwXV1dLCAib3V0IjogW1szLCAxLCAyLCA0XV19LCAiNC5BMiI6IHsiYXJncyI6IFtbWzEsIDAsIDEsIDEsIDAsIDAsIDEsIDAsIDEsIDBdLCBbMSwgMCwgMCwgMSwgMSwgMCwgMSwgMCwgMCwgMF1dXSwgIm91dCI6IFtbMC43NSwgMC42LCAwLjY2NjY2NjY2NjY2NjY2NjVdXX0sICI0LkIiOiBbMC45OCwgMC4wXSwgIjQuQyI6IFswLjg4MzcyMDkzMDIzMjU1ODIsIDAuOTUsIDAuOTg0MTI2OTg0MTI2OTg0MSwgMC43NzVdLCAiNC5EIjogMC44ODg4ODg4ODg4ODg4ODg4LCAiNS5BIjogWzAuMCwgMS4wLCAyLjAsIDQuMCwgMy4wXSwgIjUuQiI6IDk4LjE2Mzg1OTU5NzYwNjksICI1LkMiOiAtMC41NDU3MzA1NzIwOTkyOTQ3fQ=="))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Warm-up: notebook 01a in three questions

**W1.** A model has $w = (1, -2)$ and $b = 0.5$. What probability does it give to $x = (3, 1)$? Store
it in `w1_answer`.

**W2.** Two examples: the first has $y = 1$ and $h = 0.8$; the second has $y = 0$ and $h = 0.4$. What
is their (average) log-loss? Store it in `w2_answer`.

**W3.** One example has $x = (2, -1)$ and $y = 0$, and the model predicts $h = 0.7$. Using the gradient
formula from 01a, compute $\frac{\partial \ell}{\partial w_1}$, $\frac{\partial \ell}{\partial w_2}$ and
$\frac{\partial \ell}{\partial b}$. Store them in the list `w3_answer`.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# YOUR CODE HERE
w1_answer = None
w2_answer = None
w3_answer = None
check("W1", w1_answer)
check("W2", w2_answer)
check("W3", w3_answer)

*Think about W3:* all three gradient components are driven by $h - y = 0.7$. The model gave too
high a probability to a negative example, so gradient descent will push the score $w^T x + b$ **down**
for this example.

## 1. Logistic regression in 2D, with scikit-learn

Our running example for this notebook: a clinic runs **two blood tests**, A and B, on each patient.
We have 200 past patients whose diagnosis is known: $y = 1$ means **sick**, $y = 0$ means **healthy**.
So $n = 2$ and $N = 200$.

In [ ]:
def make_patients(n, seed):
    # Two blood-test results per patient; y = 1 means the patient has the disease.
    rng = np.random.default_rng(seed)
    y = (rng.uniform(size=n) < 0.4).astype(int)                     # about 40% are sick
    healthy = rng.normal([3.5, 3.0], 1.3, size=(n, 2))
    sick = rng.normal([6.0, 5.0], 1.3, size=(n, 2))
    X = np.where(y[:, None] == 1, sick, healthy)
    return np.round(X, 2), y

X_train, y_train = make_patients(200, seed=0)

print("X_train shape:", X_train.shape, "   y_train shape:", y_train.shape)
print("first 5 patients:\n", X_train[:5], "\nlabels:", y_train[:5])
print("fraction sick:", y_train.mean())

In [ ]:
def plot_patients(X, y, ax=None, title=""):
    ax = ax or plt.gca()
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k", label="healthy (y=0)", zorder=3)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red", edgecolor="k", label="sick (y=1)", zorder=3)
    ax.set_xlabel("test A  (x1)")
    ax.set_ylabel("test B  (x2)")
    ax.set_title(title)

plot_patients(X_train, y_train, title="Training data: 200 patients")
plt.legend()
plt.show()

Sick patients tend to have higher values on both tests, but the two groups **overlap**. No line
separates them perfectly, which is exactly the situation where probabilities are useful.

In 01a we trained logistic regression ourselves with gradient descent. For real work we use
**scikit-learn**, the same library you used for linear regression. Its `LogisticRegression` minimizes
the same log-loss (with a smarter optimizer than plain gradient descent):

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)

w = clf.coef_[0]          # the weights  w = (w1, w2)
b = clf.intercept_[0]     # the intercept b
print("w =", w)
print("b =", b)

A few things to know about this API:

- `clf.coef_` has shape `(1, n)`, one row of weights. `clf.coef_[0]` is our vector $w$.
  `clf.intercept_[0]` is $b$.
- `clf.predict_proba(X)` returns **two columns**: $P(y = 0 \mid x)$ and $P(y = 1 \mid x)$. They add up to
  1. We almost always want the second column, `clf.predict_proba(X)[:, 1]`, which is $h(x)$.
- `clf.predict(X)` returns classes (0 or 1). More about how it decides in section 3.
- **Note:** by default, sklearn's `LogisticRegression` adds a small extra term to the loss, called
  **regularization**. We'll explain it properly in notebook 03b. For now, you can ignore it.

In [ ]:
print("predict_proba for the first 3 patients:\n", clf.predict_proba(X_train[:3]).round(3))
print("\nP(sick) for the first 3 patients:", clf.predict_proba(X_train[:3])[:, 1].round(3))
print("predicted classes:              ", clf.predict(X_train[:3]))

**Try it yourself, before reading on:** both weights are positive. What does that tell you about
how each test result relates to being sick?

### Exercise 1.A: computing a probability yourself

A new patient has test results $x = (5, 4)$. Using `w`, `b` and your knowledge from 01a (not
`predict_proba`), compute $P(\text{sick})$ for this patient. Store it in `p_new`. The check compares it
with what sklearn says.

In [ ]:
x_new = np.array([5, 4])
# YOUR CODE HERE
p_new = None
print("your answer:", p_new)
print("sklearn:    ", clf.predict_proba([x_new])[0, 1])
check_against("1.A", p_new, clf.predict_proba([x_new])[0, 1])

### Exercise 1.B: drawing the decision boundary

In 01a (one feature) the model said "50/50" at the point where $wx + b = 0$. With two features, the
**decision boundary** is the set of all points where
$$w^T x + b = w_1 x_1 + w_2 x_2 + b = 0$$
That's the equation of a **straight line**.

1. Solve the equation for $x_2$ (as a function of $x_1$).
2. Compute `x2_at_4`, the $x_2$ value of the boundary line at $x_1 = 4$.
3. Draw the line over the data: fill in `x2_line` (the boundary's $x_2$ for every value in `x1_line`).

In [ ]:
x1_line = np.linspace(-1, 11, 100)
# YOUR CODE HERE
x2_at_4 = None
x2_line = None
check("1.B", x2_at_4, tol=5e-3)

if x2_line is not None:
    plot_patients(X_train, y_train, title="The decision boundary")
    plt.plot(x1_line, x2_line, "k-", linewidth=2.5, label="boundary  w^T x + b = 0")
    plt.xlim(-1, 11); plt.ylim(-3, 10)
    plt.legend(loc="upper left")
    plt.show()

The line goes right through the region where the two groups overlap. Patients above and to the
right of it get $P(\text{sick}) > 0.5$; patients below and to the left get $P(\text{sick}) < 0.5$.

## 2. The probability map

The boundary shows only where $h(x) = 0.5$. But the model gives a probability to **every** point of
the plane. Let's colour the plane by the predicted probability, and also draw the lines where it
equals 0.1 and 0.9.

In [ ]:
def plot_probability_map(w, b, X, y, ax=None, title="", levels=(0.1, 0.5, 0.9)):
    ax = ax or plt.gca()
    g1, g2 = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    probs = sigmoid(w[0] * g1 + w[1] * g2 + b)
    cs = ax.contourf(g1, g2, probs, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.7)
    lines = ax.contour(g1, g2, probs, levels=list(levels), colors="k",
                       linewidths=[1, 2.5, 1] if len(levels) == 3 else 1.5, linestyles="--")
    ax.clabel(lines, fmt="%.1f")
    plot_patients(X, y, ax=ax, title=title)
    return cs

cs = plot_probability_map(w, b, X_train, y_train, title="P(sick) predicted by the model")
plt.colorbar(cs, label="P(sick)")
plt.show()

Some things to notice:

- The contour lines for 0.1, 0.5 and 0.9 are **parallel straight lines**. Each one is a set of points
  where $w^T x + b$ equals some constant (for 0.9 the constant is $\log\frac{0.9}{0.1} = \log 9$).
- The probability changes only **across** the lines, never along them: along a line, $w^T x + b$ stays
  the same. The direction in which the probability grows fastest is the direction of the vector $w$.
- **The farther a point is from the boundary, the more confident the model is.** In fact
  $\frac{w^T x + b}{\|w\|}$ is exactly the signed distance of $x$ from the boundary line, so the score
  $w^T x + b$ is the distance multiplied by $\|w\|$.

That last point means **the length $\|w\|$ controls how sharp the transition is**. The slider
multiplies both $w$ and $b$ by a factor $c$: the boundary stays in place, but the band between 0.1 and
0.9 gets narrower or wider (like the zoom in 01a).

In [ ]:
def plot_sharpness(c=1.0):
    plt.figure(figsize=(6, 4.5))
    plot_probability_map(c * w, c * b, X_train, y_train, title=f"w and b multiplied by c = {c:.2f}")
    plt.show()

if HAVE_WIDGETS:
    interact(plot_sharpness, c=FloatLogSlider(value=1, base=10, min=-1, max=1, step=0.05))
else:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for ax, c in zip(axes, [0.3, 1, 5]):
        plot_probability_map(c * w, c * b, X_train, y_train, ax=ax, title=f"w and b multiplied by c = {c}")
    plt.show()

### Exercise 2.A: the 0.9 line

Find the equation of the line on which the model gives exactly $P(\text{sick}) = 0.9$, and compute
`x2_09_at_4`, the $x_2$ value of that line at $x_1 = 4$.

*Hint:* first find the value of $w^T x + b$ that gives probability 0.9 (log-odds, from 01a).

In [ ]:
# YOUR CODE HERE
x2_09_at_4 = None
check("2.A", x2_09_at_4, tol=5e-3)

### Exercise 2.B (a twist): doubling the parameters

Suppose we replace $w$ by $2w$ and $b$ by $2b$.

1. Where is the new decision boundary? Compute its $x_2$ at $x_1 = 4$: `x2_doubled_at_4`.
2. What probability does the new model give to the patient $x = (5, 4)$ from exercise 1.A? Store it in
   `p_doubled`. Is it closer to 0 or 1 than before?

Think first, then compute.

In [ ]:
# YOUR CODE HERE
x2_doubled_at_4 = None
p_doubled = None
print(x2_doubled_at_4, p_doubled)
check("2.B", [x2_doubled_at_4, p_doubled], tol=5e-3)

**Your answer:** *(write here)*

## 3. From probability to a decision: the threshold

Often we must finally **decide**: send the patient for further treatment, or not? To turn a
probability into a class we choose a **threshold** $t$:

$$\text{predict } 1 \text{ if } h(x) \ge t, \qquad \text{predict } 0 \text{ otherwise}$$

`clf.predict` uses $t = 0.5$. Let's confirm that:

In [ ]:
p_train = clf.predict_proba(X_train)[:, 1]
same = np.all(clf.predict(X_train) == (p_train >= 0.5).astype(int))
print("clf.predict is the same as (probability >= 0.5):", same)

**But 0.5 isn't always the right threshold.** Think about the clinic:

- If we tell a **sick** patient they're healthy (a *missed* case), the disease goes untreated. That can
  be very dangerous.
- If we tell a **healthy** patient they might be sick (a *false alarm*), they take another test. That's
  annoying and costs money, but it's much less harmful.

When one kind of mistake is worse than the other, we move the threshold. A **lower** $t$ flags more
patients as sick, so there are fewer missed cases but more false alarms. A **higher** $t$ does the
opposite.

**Geometrically**, $h(x) \ge t$ means $w^T x + b \ge \log\frac{t}{1-t}$. Every threshold gives a line
**parallel** to the boundary:

In [ ]:
plot_probability_map(w, b, X_train, y_train, levels=(0.2, 0.5, 0.8),
                     title="Thresholds 0.2, 0.5 and 0.8: three parallel lines")
plt.show()

### Exercise 3.A: predicting with a threshold, and counting mistakes

1. Write `predict_with_threshold(p, t)`, which takes an array of probabilities `p` and a threshold
   `t`, and returns an array of 0/1 predictions.
2. For every threshold in `thresholds` (0.05, 0.10, ..., 0.95), count on the **training** data:
   - `missed[i]`: sick patients predicted healthy (missed cases);
   - `false_alarms[i]`: healthy patients predicted sick.
3. Plot both counts against the threshold (the plotting code is given).

In [ ]:
def predict_with_threshold(p, t):
    # YOUR CODE HERE
    return None

check_fn("3.A", predict_with_threshold)

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
# YOUR CODE HERE
missed = None
false_alarms = None
check("3.A2", None if missed is None else [list(missed), list(false_alarms)])

if missed is not None:
    plt.plot(thresholds, missed, "o-", color="tab:red", label="missed sick patients")
    plt.plot(thresholds, false_alarms, "o-", color="tab:blue", label="false alarms")
    plt.xlabel("threshold t"); plt.ylabel("number of patients")
    plt.legend(); plt.title("The trade-off: moving the threshold")
    plt.show()

### Exercise 3.B: choosing a threshold

The clinic's rule: **at most 2 sick patients may be missed** (on the training data). Among the
thresholds in `thresholds`, which is the **largest** one that satisfies the rule? (Why the largest?
Because it keeps the number of false alarms as low as possible.) Store it in `chosen_t`, and print how
many false alarms it causes.

In [ ]:
# YOUR CODE HERE
chosen_t = None
print("chosen threshold:", chosen_t)
check("3.B", chosen_t)

### Exercise 3.C (a twist): where is the line for your threshold?

For the threshold $t = 0.2$, compute $x_2$ of the corresponding line at $x_1 = 4$. Store it in
`x2_t02_at_4`. Is the line above or below the 0.5 boundary? Does that make sense?

In [ ]:
# YOUR CODE HERE
x2_t02_at_4 = None
check("3.C", x2_t02_at_4, tol=5e-3)

**Your answer:** *(write here)*

## 4. How good is the classifier?

### 4.1 Test on new data

To know how well a model works on **future** patients, we must test it on patients it **did not see**
during training. A model can look great on its own training data just by fitting its quirks. In this
notebook we simply generate 200 new patients from the same source. (In notebook 02 we'll see how to
split one dataset into a training part and a test part.)

In [ ]:
X_test, y_test = make_patients(200, seed=1)      # 200 NEW patients, never seen in training
p_test = clf.predict_proba(X_test)[:, 1]
pred_test = clf.predict(X_test)                  # threshold 0.5

print("test accuracy:", np.mean(pred_test == y_test))

### 4.2 The confusion matrix

Accuracy is a single number that hides **which** mistakes the model makes. The **confusion matrix**
counts all four combinations of truth and prediction:

| | predicted healthy (0) | predicted sick (1) |
|---|---|---|
| **actually healthy (0)** | **TN**: true negative | **FP**: false positive (false alarm) |
| **actually sick (1)** | **FN**: false negative (missed case) | **TP**: true positive |

("Positive" means "predicted 1", and "true/false" says whether that prediction was right.)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred_test)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

sklearn prints the matrix with the **true** class as rows and the **predicted** class as columns, in
the same layout as the table above.

### 4.3 Accuracy, precision, recall, F1

From the four counts we build the standard measures:

- **Accuracy** $= \dfrac{TP + TN}{TP + TN + FP + FN}$: the fraction of all predictions that are right.
- **Precision** $= \dfrac{TP}{TP + FP}$: *of the patients we flagged as sick, how many really are?*
  (High precision means few false alarms.)
- **Recall** $= \dfrac{TP}{TP + FN}$: *of the patients who really are sick, how many did we catch?*
  (High recall means few missed cases.) Also called *sensitivity*, or the *true positive rate*.
- **F1** $= \dfrac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$:
  a single number that is high only if **both** precision and recall are high. (It's their
  *harmonic mean*: if either one is low, F1 is low.)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print(f"accuracy  = {accuracy_score(y_test, pred_test):.3f}")
print(f"precision = {precision_score(y_test, pred_test):.3f}")
print(f"recall    = {recall_score(y_test, pred_test):.3f}")
print(f"F1        = {f1_score(y_test, pred_test):.3f}")
print()
print(classification_report(y_test, pred_test, target_names=["healthy", "sick"]))

`classification_report` prints precision, recall and F1 **for each class** (treating each one in
turn as the "positive" class). The row we computed above is the `sick` row. `support` is the number of
test patients in each class.

### Exercise 4.A: compute the measures yourself

1. Write `confusion_counts(y_true, y_pred)`, which returns `[TP, FP, FN, TN]` for two 0/1 arrays.
2. Write `my_metrics(y_true, y_pred)`, which returns `[precision, recall, f1]`. Use your
   `confusion_counts`.
3. The last lines compare your functions with sklearn on the test data.

In [ ]:
def confusion_counts(y_true, y_pred):
    # YOUR CODE HERE
    return None

def my_metrics(y_true, y_pred):
    # YOUR CODE HERE
    return None

check_fn("4.A", confusion_counts)
check_fn("4.A2", my_metrics)
print("yours on the test set:  ", my_metrics(y_test, pred_test))
print("sklearn on the test set:", [precision_score(y_test, pred_test), recall_score(y_test, pred_test),
                                   f1_score(y_test, pred_test)])

### Exercise 4.B: when accuracy lies

A rare disease affects **2%** of people. Here are 1000 people, 20 of them sick. A lazy "classifier"
predicts **healthy for everyone**. Compute its accuracy and its recall (`acc_lazy`, `recall_lazy`).

In [ ]:
y_rare = np.zeros(1000, dtype=int)
y_rare[:20] = 1
pred_lazy = np.zeros(1000, dtype=int)          # "everyone is healthy"
# YOUR CODE HERE
acc_lazy = None
recall_lazy = None
print("accuracy:", acc_lazy, "  recall:", recall_lazy)
check("4.B", [acc_lazy, recall_lazy])

**Your answer:** *(write here)*

### Exercise 4.C: how the threshold changes the measures

Compute precision and recall on the **test** set at threshold 0.3 and at threshold 0.7. Store them in
`prec_03, rec_03, prec_07, rec_07`. Which threshold would you choose for the clinic, and why?

In [ ]:
# YOUR CODE HERE
prec_03, rec_03, prec_07, rec_07 = None, None, None, None
print(f"t = 0.3: precision = {prec_03}, recall = {rec_03}")
print(f"t = 0.7: precision = {prec_07}, recall = {rec_07}")
check("4.C", [prec_03, rec_03, prec_07, rec_07], tol=5e-3)

**Your answer:** *(write here)*

### 4.4 The ROC curve and AUC: all thresholds at once

Precision and recall depend on the threshold. Can we judge the model **itself**, independent of any
particular threshold? Yes. For every possible threshold, compute two rates:

- **True positive rate** (TPR, the same as recall) $= \dfrac{TP}{TP + FN}$: the fraction of sick
  patients we catch;
- **False positive rate** (FPR) $= \dfrac{FP}{FP + TN}$: the fraction of healthy patients we wrongly
  flag.

Plot TPR against FPR for all thresholds, and you get the **ROC curve**. (ROC stands for "receiver
operating characteristic", a name from radar engineering.)

- Threshold 1: nobody is flagged, which is the point $(0, 0)$.
- Threshold 0: everybody is flagged, which is the point $(1, 1)$.
- A **perfect** model passes through $(0, 1)$: it catches everyone with no false alarms.
- A model that **guesses randomly** lies on the diagonal.

The **AUC** ("area under the curve") summarizes the whole curve in one number: 1.0 is perfect, and 0.5
is no better than guessing.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresholds = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, linewidth=2.5, label=f"our model (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k:", label="random guessing (AUC = 0.5)")
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
plt.plot(fp / (fp + tn), tp / (tp + fn), "ko", markersize=9, label="threshold 0.5")
plt.xlabel("false positive rate (FPR)")
plt.ylabel("true positive rate (TPR = recall)")
plt.title("ROC curve on the test set")
plt.legend(loc="lower right")
plt.show()

AUC also has a lovely direct meaning: **pick one random sick patient and one random healthy
patient. The AUC is the probability that the model gives the sick one a higher score.** It measures how
well the model **ranks** patients, independently of any threshold. You'll verify this meaning in
exercise 4.D.

**Now play with the threshold.** The slider moves the threshold; watch the line move on the data, the
confusion matrix change, and the point slide along the ROC curve.

In [ ]:
def show_threshold(t=0.5):
    pred = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    plot_probability_map(w, b, X_test, y_test, ax=axes[0], levels=(t,), title=f"test data, threshold t = {t:.2f}")
    axes[1].plot(fpr, tpr, linewidth=2)
    axes[1].plot([0, 1], [0, 1], "k:")
    axes[1].plot(fp / (fp + tn), tp / (tp + fn), "ro", markersize=11)
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR (recall)")
    axes[1].set_title("where this threshold sits on the ROC curve")
    plt.show()
    prec = tp / (tp + fp) if tp + fp > 0 else float("nan")
    print(f"confusion matrix:  TN = {tn:3d}   FP = {fp:3d}")
    print(f"                   FN = {fn:3d}   TP = {tp:3d}")
    print(f"accuracy = {(tp + tn) / len(y_test):.3f}   precision = {prec:.3f}   recall = {tp / (tp + fn):.3f}")

if HAVE_WIDGETS:
    interact(show_threshold, t=FloatSlider(value=0.5, min=0.02, max=0.98, step=0.02, description="threshold"))
else:
    for t in [0.2, 0.5, 0.8]:
        show_threshold(t)

### Exercise 4.D: AUC by counting pairs

Here are 6 patients with their true labels and the model's scores:

| patient | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $y$ | 1 | 1 | 0 | 1 | 0 | 0 |
| score | 0.9 | 0.7 | 0.6 | 0.4 | 0.3 | 0.1 |

1. **By hand:** there are $3 \times 3 = 9$ (sick, healthy) pairs. In how many does the sick patient get
   the higher score? What fraction is that?
2. **In code:** write `auc_by_pairs(y, scores)`, which computes this fraction for any data. Check it on
   the table (store the result in `auc_small`), then compare it with `roc_auc_score` on the whole test
   set.

In [ ]:
def auc_by_pairs(y, scores):
    # YOUR CODE HERE
    return None

y_small = np.array([1, 1, 0, 1, 0, 0])
s_small = np.array([0.9, 0.7, 0.6, 0.4, 0.3, 0.1])
auc_small = auc_by_pairs(y_small, s_small)
print("your AUC on the small table:", auc_small)
check("4.D", auc_small)
print("your AUC on the test set:   ", auc_by_pairs(y_test, p_test))
print("sklearn roc_auc_score:      ", roc_auc_score(y_test, p_test))

**Your answer:** *(write here)*

## 5. The general case: many features

Real datasets rarely have just 2 features. Logistic regression works **exactly the same way** with
$n$ features: $w$ is just a longer vector, and $h(x) = \sigma(w^T x + b)$. The boundary
$w^T x + b = 0$ is no longer a line but a **hyperplane**: the $n$-dimensional version of a line
(a plane in 3D, and so on).

The catch: we can't draw it. We can't plot in 5D! Two useful tricks help.

1. **Read the learned weights.** The sign and size of $w_j$ tell us how feature $j$ pushes the
   prediction: $w_j > 0$ pushes toward class 1, $w_j < 0$ toward class 0, and a bigger $|w_j|$ pushes
   harder. Increasing $x_j$ by 1 adds $w_j$ to the log-odds (01a). **This comparison is only fair when
   the features are on comparable scales** (exercise 5.B shows why).
2. **Look at a 2D slice.** Fix all features except two at their average value, and plot the boundary in
   just those two. It's still a straight line in the slice, because the full boundary is a hyperplane.

Let's try both on a synthetic dataset with 5 features. Because we generate it ourselves, we **know the
true weights** and can check whether the model recovers them.

In [ ]:
rng = np.random.default_rng(5)
w_true = np.array([2.0, -1.5, 1.0, 0.0, 0.5])      # hidden from the learner!
b_true = -0.5
X5 = rng.normal(size=(1000, 5))                    # 5 features, all on the same scale
y5 = (rng.uniform(size=1000) < 1 / (1 + np.exp(-(X5 @ w_true + b_true)))).astype(int)

clf5 = LogisticRegression().fit(X5, y5)
print("true weights:   ", w_true)
print("learned weights:", clf5.coef_[0].round(2))
print("true b:", b_true, "   learned b:", clf5.intercept_[0].round(2))

In [ ]:
plt.figure(figsize=(7, 3.5))
idx = np.arange(5)
plt.bar(idx - 0.2, w_true, width=0.4, label="true")
plt.bar(idx + 0.2, clf5.coef_[0], width=0.4, label="learned")
plt.axhline(0, color="k", linewidth=0.8)
plt.xticks(idx, [f"x{j + 1}" for j in idx])
plt.ylabel("weight")
plt.legend()
plt.title("True vs. learned weights")
plt.show()

The learned weights are close to the true ones: same signs, similar sizes. Feature `x4`, which has
true weight 0 (it has **no** effect on $y$), gets a learned weight close to 0. The small differences
come from having a finite sample of 1000 examples, and from the regularization sklearn adds (03b).

### Exercise 5.A: ranking the features

Rank the 5 features from the **most** influential to the **least** influential, according to the
learned weights. Store the list of **indices** (0 to 4) in `ranking`. For example, `[3, 0, ...]` would
mean "`x4` is the most influential, then `x1`, ...".

*Hint:* influence is about the size of the weight, not its sign. `np.argsort` sorts from small to
large.

Then, in words: which feature pushes toward class 0?

In [ ]:
# YOUR CODE HERE
ranking = None
print("ranking:", ranking)
check("5.A", ranking)

**Your answer:** *(write here)*

### Exercise 5.B (a twist): why the scales must be comparable

Suppose feature `x1` had been measured in different units, say centimeters instead of meters, so all
its values are **100 times larger**. Nothing about the *information* has changed.

1. Make a copy of `X5` with the first column multiplied by 100, and fit a new `LogisticRegression` on it.
2. Compute `ratio` = (learned weight of `x1` before) / (learned weight of `x1` after).
3. Look at the new weights: according to them, which feature now looks the *least* important?

In [ ]:
# YOUR CODE HERE
ratio = None
print("ratio:", ratio)
check("5.B", ratio, tol=0.5)

**Your answer:** *(write here)*

### 5.1 A 2D slice

Now the second trick. We choose two features, fix the other three at their **average**, and draw the
probability map in the plane of the two chosen features. In the slice, the model is
$$h = \sigma\big(w_i x_i + w_j x_j + \underbrace{b + \textstyle\sum_{k \ne i, j} w_k \bar{x}_k}_{\text{a constant}}\big)$$
where $\bar{x}_k$ is the average of feature $k$. It's a 2-feature model again, with a **new intercept**
that collects all the fixed features.

### Exercise 5.C: the intercept of the slice

We'll slice through the two **most** influential features (`ranking[:2]` from 5.A, which are `x1` and
`x2`). Compute `b_slice`, the constant in the formula above, using `clf5.coef_[0]`,
`clf5.intercept_[0]` and the column averages of `X5`. The plotting cell below then draws the slice.

In [ ]:
i, j = 0, 1                          # the two features of the slice: x1 and x2
others = [k for k in range(5) if k not in (i, j)]
w5, b5 = clf5.coef_[0], clf5.intercept_[0]
# YOUR CODE HERE
b_slice = None
print("b_slice:", b_slice)
check("5.C", b_slice, tol=5e-3)

In [ ]:
if b_slice is not None:
    g1, g2 = np.meshgrid(np.linspace(-3.5, 3.5, 300), np.linspace(-3.5, 3.5, 300))
    probs = sigmoid(w5[i] * g1 + w5[j] * g2 + b_slice)
    plt.figure(figsize=(6, 5))
    cs = plt.contourf(g1, g2, probs, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.7)
    plt.contour(g1, g2, probs, levels=[0.5], colors="k", linewidths=2.5)
    plt.scatter(X5[y5 == 0, i], X5[y5 == 0, j], s=8, c="tab:blue", alpha=0.5)
    plt.scatter(X5[y5 == 1, i], X5[y5 == 1, j], s=8, c="tab:red", alpha=0.5)
    plt.colorbar(cs, label="P(y = 1)")
    plt.xlabel(f"x{i + 1}"); plt.ylabel(f"x{j + 1}")
    plt.title(f"2D slice: x{i + 1} and x{j + 1} free, the others at their average")
    plt.show()

The boundary in the slice is a straight line, as promised. The points on the "wrong" side of it
aren't necessarily mistakes: each point also has its own values of `x3`, `x4` and `x5`, which the slice
fixes at the average. A slice shows the model's behaviour for a "typical" example; it isn't the full
picture.

**Try it yourself:** change `i, j` to slice through `x1` and `x4` instead (indices 0 and 3), and rerun
both cells. What does the boundary look like, and why? (Remember that `x4` has a weight close to 0.)

## Recap

- **sklearn**: `LogisticRegression().fit(X, y)`.
  - `coef_[0]` is $w$ and `intercept_[0]` is $b$.
  - `predict_proba(X)[:, 1]` gives $P(y = 1 \mid x)$, and `predict` uses threshold 0.5.
  - By default it adds regularization (explained in 03b).
- **Decision boundary**: $w^T x + b = 0$. In 2D it's a line; with $n$ features, a hyperplane.
  - Lines of equal probability are parallel to it.
  - $\|w\|$ controls how sharply the probability changes across the boundary.
- **Threshold**: predict 1 if $h(x) \ge t$. Choose $t$ according to the costs of the two kinds of
  mistake: lowering it trades false alarms for fewer missed cases.
- **Evaluation**, always on **data the model didn't train on**:
  - the confusion matrix (TP, FP, FN, TN);
  - accuracy (misleading when the classes are imbalanced), precision, recall and F1;
  - the ROC curve and AUC. The AUC is threshold-free: the probability that a random positive is
    ranked above a random negative.
- **Many features**:
  - Read the signs and sizes of the weights, but only compare them when the features are on
    comparable scales.
  - Visualize with 2D slices.
- **Next (01c):** a practice set on everything so far. Then **02**: real data.

## References and further reading

**Interactive explanations (highly recommended)**

- **[ROC & AUC](https://mlu-explain.github.io/roc-auc/)**, by Jared Wilber (MLU-Explain). A beautiful
  visual, interactive walk through thresholds, the ROC curve and AUC. It also tells the story of where
  the name "receiver operating characteristic" comes from: radar operators in World War II, who had to
  decide whether a blip on the screen was an enemy aircraft or just noise. That's exactly our
  "threshold" problem!

<p align="center">
  <a href="https://mlu-explain.github.io/roc-auc/">
    <img src="https://mlu-explain.github.io/assets/ogimages/ogimage-roc-auc.png" width="600"
         alt="ROC & AUC, MLU-Explain">
  </a>
  <br>
  <sub>Image: <i>ROC &amp; AUC</i> by Jared Wilber, MLU-Explain, licensed
  <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  (The image loads when you're online.)</sub>
</p>

- **[Precision & Recall](https://mlu-explain.github.io/precision-recall/)**, by Jared Wilber
  (MLU-Explain): precision, recall and the F1-score, visually.
- **[Logistic Regression](https://mlu-explain.github.io/logistic-regression/)**, by Erin Bugbee and
  Jared Wilber (MLU-Explain): a visual review of notebooks 01a and 01b.

**Short readings**

- Google Machine Learning Crash Course:
  [Thresholds and the confusion matrix](https://developers.google.com/machine-learning/crash-course/classification/thresholding)
  and [ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc).
  Clear, short pages with small interactive exercises.

**Videos**

- StatQuest with Josh Starmer:
  [ROC and AUC, Clearly Explained!](https://www.youtube.com/watch?v=4jRBRDbJemM), and the
  [Logistic Regression playlist](https://www.youtube.com/playlist?list=PLblh5JKOoLUKxzEP5HA2d-Li7IJkHfXSe).

**Documentation (for looking things up)**

- scikit-learn:
  [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
  and the [model evaluation guide](https://scikit-learn.org/stable/modules/model_evaluation.html),
  which covers the [confusion matrix](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix),
  [precision, recall and F1](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-and-f-measures),
  and ROC/AUC.